## SNS & the fan-out pattern

SNS is fully managed **pub/sub**: a publisher sends to a **topic** and SNS delivers a copy to **every subscriber** at once. There's **no queue** — once delivered the message is gone from SNS; **persistence is the subscriber's job**. Subscribers can be **SQS queues** (by far the most common), **Lambda**, **HTTP/HTTPS** webhooks, **email/SMS/mobile push**, or **Firehose** (into S3/Redshift/OpenSearch). A topic can have up to **12.5 M subscriptions**; messages cap at 256 KB. **Filter policies** let each subscriber declare what it cares about (by attributes or body), so it only receives matching messages instead of every one.

The **SNS + SQS fan-out** is the **most reused pattern in AWS messaging**. One event must reach several downstream systems, each at its own pace with its own retries. Publishing directly to multiple queues would couple the producer to the topology (add a consumer → change the producer; one write fails → partial delivery). Instead: the producer publishes **once** to an SNS topic; SNS delivers to each subscribed **SQS queue**; each queue feeds an **independent consumer** with its own DLQ, visibility timeout, and scaling.

The producer **doesn't know who's downstream** — adding a consumer is one new queue + one subscription, no producer change, and each consumer survives its peers' failures. Canonical: an order that must email the customer, charge the card, reserve inventory, and notify shipping.